In [1]:
#Window attention

import torch
import os
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader,random_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [18]:
from torch.cuda import manual_seed
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/Breast_Cancer/dataset_cancer_v1/classificacao_binaria/40X'


transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(path,transform = transform)

train_size = int(0.8*len(full_dataset))
val_size = int(0.1*len(full_dataset))
test_size = len(full_dataset) - train_size - val_size


train_dataset,val_dataset,test_dataset = random_split(
    full_dataset,
    [train_size,val_size,test_size],
    generator= torch.Generator().manual_seed(42)
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
#Data Loader

train_loader = DataLoader(
    train_dataset,
    batch_size= 32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [20]:
class WindowAttention(nn.Module):
    def __init__(self, dim=512, window_size=8, num_heads=4):
        super().__init__()
        self.dim         = dim
        self.window_size = window_size
        self.num_heads   = num_heads

        assert dim % window_size == 0, "dim must be divisible by window_size"

        self.head_dim = window_size // num_heads
        assert self.head_dim >= 1, "window_size must be >= num_heads"

        # Project each window into Q, K, V
        self.qkv  = nn.Linear(window_size, window_size * 3, bias=False)
        self.proj = nn.Linear(window_size, window_size)

        self.scale = self.head_dim ** -0.5
        self.norm  = nn.LayerNorm(dim)

    def forward(self, x):
        B, C = x.shape
        residual = x

        # ── partition into windows ──────────────────────────────────────────
        # (B, C) → (B, num_windows, window_size)
        num_windows = C // self.window_size
        x = x.view(B, num_windows, self.window_size)

        # ── QKV projection ───────────────────────────────────────────────────
        qkv = self.qkv(x)                          # (B, W, 3*ws)
        qkv = qkv.reshape(B, num_windows,
                           3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)          # (3, B, heads, W, head_dim)
        q, k, v = qkv.unbind(0)                    # each: (B, heads, W, head_dim)

        # ── Scaled dot-product attention ─────────────────────────────────────
        attn = (q @ k.transpose(-2, -1)) * self.scale   # (B, heads, W, W)
        attn = attn.softmax(dim=-1)

        x = (attn @ v)
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(B, num_windows, self.window_size)


        x = self.proj(x)
        x = x.reshape(B, C)


        x = self.norm(x + residual)
        return x

In [21]:
class ViTWithWindowAttention(nn.Module):
    def __init__(self, num_classes=2, window_size=8, num_heads=4):
        super().__init__()

        self.vit = vit_b_16(weights="IMAGENET1K_V1")
        in_features = self.vit.heads.head.in_features

        self.vit.heads = nn.Linear(in_features, 512)

        self.attention  = WindowAttention(dim=512,
                                          window_size=window_size,
                                          num_heads=num_heads)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.vit(x)
        x = self.attention(x)
        x = self.classifier(x)
        return x


model = ViTWithWindowAttention(num_classes=2, window_size=8, num_heads=4).to(device)
print(model)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 173MB/s]


ViTWithWindowAttention(
  (vit): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          (l

In [22]:
import torch.optim

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)

In [23]:
epochs = 10
best_loss = float("inf")

for epoch in range(epochs):
    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    training_loss = running_loss / len(train_loader)
    train_acc = correct / total * 100

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    validation_loss = val_loss / len(val_loader)
    val_acc = correct / total * 100

    print(f"Epoch {epoch+1}/{epochs}, "
          f"Train Loss: {training_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {validation_loss:.4f}, Val Acc: {val_acc:.2f}%")

    if validation_loss < best_loss:
        best_loss = validation_loss
        torch.save(model.state_dict(), "best_model.pth")

Epoch 1/10, Train Loss: 0.6199, Train Acc: 71.18%, Val Loss: 0.4781, Val Acc: 78.89%
Epoch 2/10, Train Loss: 0.3191, Train Acc: 86.53%, Val Loss: 0.2500, Val Acc: 89.95%
Epoch 3/10, Train Loss: 0.1347, Train Acc: 94.55%, Val Loss: 0.3612, Val Acc: 90.95%
Epoch 4/10, Train Loss: 0.0789, Train Acc: 96.99%, Val Loss: 0.1670, Val Acc: 95.98%
Epoch 5/10, Train Loss: 0.0724, Train Acc: 97.31%, Val Loss: 0.1349, Val Acc: 94.97%
Epoch 6/10, Train Loss: 0.0715, Train Acc: 97.62%, Val Loss: 0.2316, Val Acc: 93.97%
Epoch 7/10, Train Loss: 0.0355, Train Acc: 99.12%, Val Loss: 0.1924, Val Acc: 94.97%
Epoch 8/10, Train Loss: 0.0179, Train Acc: 99.25%, Val Loss: 0.2013, Val Acc: 93.47%
Epoch 9/10, Train Loss: 0.0109, Train Acc: 99.56%, Val Loss: 0.3151, Val Acc: 95.98%
Epoch 10/10, Train Loss: 0.0520, Train Acc: 98.25%, Val Loss: 0.1059, Val Acc: 95.48%


In [24]:
model.load_state_dict(torch.load("best_model.pth"))

<All keys matched successfully>

In [25]:
#Accurcy

model.eval()
correct =0
total = 0

with torch.no_grad():

  for images,labels in test_loader:

    images = images.to(device)
    labels = labels.to(device)

    output = model(images)

    _, predicated = torch.max(output,1)

    total+= labels.size(0)

    correct += (predicated == labels).sum().item()

test_accy = 100*correct / total

print(f"Test Accuracy: {test_accy:.4f}")


Test Accuracy: 96.0000
